# 📊 Embryoscope Embryo Metrics: DuckDB vs AWS Athena

This notebook connects to both the local **DuckDB** database and **AWS Athena (Production)** to execute data quality queries on the `embryoscope_embrioes` table. It retrieves:
1. Total row count
2. Unique embryo IDs
3. Unique patient IDs
4. Unique patient IDxs
5. Unique `prontuario` (medical record numbers)
6. Percentage of patients/records with a valid `prontuario` (not null or -1)

Metrics are calculated overall, per year (`embryo_EmbryoDate`), and per clinic location.

In [1]:
import duckdb
import pandas as pd
from pyathena import connect

# Connections setup
DUCKDB_PATH = '../../database/huntington_data_lake.duckdb'
print("Connecting to DuckDB...")
duck_con = duckdb.connect(DUCKDB_PATH, read_only=True)

print("Connecting to AWS Athena...")
ath_con = connect(
    region_name="sa-east-1",
    work_group="datalake-admins"
)
ath_cur = ath_con.cursor()

Connecting to DuckDB...
Connecting to AWS Athena...


## 🔍 Part 1: DuckDB Queries (Local Gold Layer)
Running local metrics on `gold.embryoscope_embrioes` including `patient_PatientIDx`.

In [2]:
# 1. DuckDB Overall
duck_overall_q = """
SELECT 
    'TOTAL' as grouping_type,
    'ALL' as grouping_value,
    COUNT(*) as total_rows,
    COUNT(DISTINCT embryo_EmbryoID) as unique_embryos,
    COUNT(DISTINCT patient_PatientID) as unique_patients,
    COUNT(DISTINCT patient_PatientIDx) as unique_patient_idxs,
    COUNT(DISTINCT prontuario) as unique_prontuarios,
    ROUND(COUNT(DISTINCT CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN patient_PatientID END) * 100.0 / COUNT(DISTINCT patient_PatientID), 2) as pct_patients_with_valid_prontuario,
    ROUND(COUNT(CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN 1 END) * 100.0 / COUNT(*), 2) as pct_rows_with_valid_prontuario
FROM gold.embryoscope_embrioes;
"""
df_duck_overall = duck_con.execute(duck_overall_q).df()
print("--- DuckDB Overall Totals ---")
display(df_duck_overall)

# 2. DuckDB Per Year
duck_year_q = """
SELECT 
    EXTRACT(YEAR FROM embryo_EmbryoDate) as embryo_year,
    COUNT(*) as total_rows,
    COUNT(DISTINCT embryo_EmbryoID) as unique_embryos,
    COUNT(DISTINCT patient_PatientID) as unique_patients,
    COUNT(DISTINCT patient_PatientIDx) as unique_patient_idxs,
    COUNT(DISTINCT prontuario) as unique_prontuarios,
    ROUND(COUNT(DISTINCT CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN patient_PatientID END) * 100.0 / COUNT(DISTINCT patient_PatientID), 2) as pct_patients_with_valid_prontuario,
    ROUND(COUNT(CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN 1 END) * 100.0 / COUNT(*), 2) as pct_rows_with_valid_prontuario
FROM gold.embryoscope_embrioes
GROUP BY 1
ORDER BY 1;
"""
df_duck_year = duck_con.execute(duck_year_q).df()
print("\n--- DuckDB Per Year ---")
display(df_duck_year)

# 3. DuckDB Per Location
duck_loc_q = """
SELECT 
    patient_unit_huntington,
    COUNT(*) as total_rows,
    COUNT(DISTINCT embryo_EmbryoID) as unique_embryos,
    COUNT(DISTINCT patient_PatientID) as unique_patients,
    COUNT(DISTINCT patient_PatientIDx) as unique_patient_idxs,
    COUNT(DISTINCT prontuario) as unique_prontuarios,
    ROUND(COUNT(DISTINCT CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN patient_PatientID END) * 100.0 / COUNT(DISTINCT patient_PatientID), 2) as pct_patients_with_valid_prontuario,
    ROUND(COUNT(CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN 1 END) * 100.0 / COUNT(*), 2) as pct_rows_with_valid_prontuario
FROM gold.embryoscope_embrioes
GROUP BY 1
ORDER BY 1;
"""
df_duck_loc = duck_con.execute(duck_loc_q).df()
print("\n--- DuckDB Per Location ---")
display(df_duck_loc)

--- DuckDB Overall Totals ---


,grouping_type,grouping_value,total_rows,unique_embryos,unique_patients,unique_patient_idxs,unique_prontuarios,pct_patients_with_valid_prontuario,pct_rows_with_valid_prontuario
0,TOTAL,ALL,152917,152917,13477,14105,13062,99.41,98.83



--- DuckDB Per Year ---


,embryo_year,total_rows,unique_embryos,unique_patients,unique_patient_idxs,unique_prontuarios,pct_patients_with_valid_prontuario,pct_rows_with_valid_prontuario
0,2017,602,602,81,81,81,100.00,100.00
1,2018,4127,4127,455,458,455,99.78,99.81
2,2019,14459,14459,1642,1671,1638,99.39,99.48
3,2020,17348,17348,1798,1823,1775,99.17,95.75
4,2021,21132,21132,2297,2345,2258,99.00,99.19
5,2022,19712,19712,2101,2144,2043,99.43,99.48
6,2023,19215,19215,1985,2008,1963,99.45,99.43
7,2024,17282,17282,1760,1776,1752,99.77,99.73
8,2025,23570,23570,2358,2397,2340,99.58,99.55
9,2026,15470,15470,1581,1596,1574,99.81,97.21



--- DuckDB Per Location ---


,patient_unit_huntington,total_rows,unique_embryos,unique_patients,unique_patient_idxs,unique_prontuarios,pct_patients_with_valid_prontuario,pct_rows_with_valid_prontuario
0,Belo Horizonte,27304,27304,2562,2562,2448,99.02,99.17
1,Brasilia,18478,18478,1791,1793,1747,98.49,95.43
2,Ibirapuera,66859,66859,6027,6154,5952,99.75,99.21
3,Salvador,2315,2315,246,246,241,97.56,97.15
4,Vila Mariana,37961,37961,3339,3350,3301,99.64,99.66


## ☁️ Part 2: AWS Athena Queries (Production Gold Layer)
Running metrics on `gold_huntington_prod.embryoscope_embrioes` including `patient_id_x`.

In [3]:
# 1. Athena Overall
ath_overall_q = """
SELECT 
    'TOTAL' as grouping_type,
    'ALL' as grouping_value,
    COUNT(*) as total_rows,
    COUNT(DISTINCT embryo_embryo_id) as unique_embryos,
    COUNT(DISTINCT patient_id) as unique_patients,
    COUNT(DISTINCT patient_id_x) as unique_patient_idxs,
    COUNT(DISTINCT prontuario) as unique_prontuarios,
    ROUND(COUNT(DISTINCT CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN patient_id END) * 100.0 / COUNT(DISTINCT patient_id), 2) as pct_patients_with_valid_prontuario,
    ROUND(COUNT(CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN 1 END) * 100.0 / COUNT(*), 2) as pct_rows_with_valid_prontuario
FROM gold_huntington_prod.embryoscope_embrioes;
"""
ath_cur.execute(ath_overall_q)
df_ath_overall = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
print("--- Athena Overall Totals ---")
display(df_ath_overall)

# 2. Athena Per Year
ath_year_q = """
SELECT 
    EXTRACT(YEAR FROM embryo_embryo_date) as embryo_year,
    COUNT(*) as total_rows,
    COUNT(DISTINCT embryo_embryo_id) as unique_embryos,
    COUNT(DISTINCT patient_id) as unique_patients,
    COUNT(DISTINCT patient_id_x) as unique_patient_idxs,
    COUNT(DISTINCT prontuario) as unique_prontuarios,
    ROUND(COUNT(DISTINCT CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN patient_id END) * 100.0 / COUNT(DISTINCT patient_id), 2) as pct_patients_with_valid_prontuario,
    ROUND(COUNT(CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN 1 END) * 100.0 / COUNT(*), 2) as pct_rows_with_valid_prontuario
FROM gold_huntington_prod.embryoscope_embrioes
GROUP BY 1
ORDER BY 1;
"""
ath_cur.execute(ath_year_q)
df_ath_year = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
print("\n--- Athena Per Year ---")
display(df_ath_year)

# 3. Athena Per Location
ath_loc_q = """
SELECT 
    unit_huntington,
    COUNT(*) as total_rows,
    COUNT(DISTINCT embryo_embryo_id) as unique_embryos,
    COUNT(DISTINCT patient_id) as unique_patients,
    COUNT(DISTINCT patient_id_x) as unique_patient_idxs,
    COUNT(DISTINCT prontuario) as unique_prontuarios,
    ROUND(COUNT(DISTINCT CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN patient_id END) * 100.0 / COUNT(DISTINCT patient_id), 2) as pct_patients_with_valid_prontuario,
    ROUND(COUNT(CASE WHEN prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1' THEN 1 END) * 100.0 / COUNT(*), 2) as pct_rows_with_valid_prontuario
FROM gold_huntington_prod.embryoscope_embrioes
GROUP BY 1
ORDER BY 1;
"""
ath_cur.execute(ath_loc_q)
df_ath_loc = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
print("\n--- Athena Per Location ---")
display(df_ath_loc)

--- Athena Overall Totals ---


,grouping_type,grouping_value,total_rows,unique_embryos,unique_patients,unique_patient_idxs,unique_prontuarios,pct_patients_with_valid_prontuario,pct_rows_with_valid_prontuario
0,TOTAL,ALL,152867,152867,13473,14101,13058,99.41,98.73



--- Athena Per Year ---


,embryo_year,total_rows,unique_embryos,unique_patients,unique_patient_idxs,unique_prontuarios,pct_patients_with_valid_prontuario,pct_rows_with_valid_prontuario
0,2017,602,602,81,81,81,100.00,100.00
1,2018,4127,4127,455,458,455,99.78,99.81
2,2019,14459,14459,1642,1671,1638,99.39,99.48
3,2020,17348,17348,1798,1823,1774,99.11,94.82
4,2021,21132,21132,2297,2345,2258,99.00,99.19
5,2022,19712,19712,2101,2144,2043,99.43,99.48
6,2023,19215,19215,1985,2008,1962,99.40,99.39
7,2024,17282,17282,1760,1776,1752,99.77,99.73
8,2025,23527,23527,2354,2393,2338,99.66,99.64
9,2026,15463,15463,1581,1596,1574,99.81,97.17



--- Athena Per Location ---


,unit_huntington,total_rows,unique_embryos,unique_patients,unique_patient_idxs,unique_prontuarios,pct_patients_with_valid_prontuario,pct_rows_with_valid_prontuario
0,Belo Horizonte,27304,27304,2562,2562,2448,99.02,99.17
1,Brasilia,18478,18478,1791,1793,1746,98.44,94.60
2,Ibirapuera,66809,66809,6023,6150,5948,99.75,99.21
3,Salvador,2315,2315,246,246,242,97.97,97.54
4,Vila Mariana,37961,37961,3339,3350,3301,99.64,99.66


## ⚖️ Part 3: Comparative Breakdown (DuckDB vs Athena Prod)
Side-by-side location and year comparisons, followed by a **Year × Unit pivot** (year extracted from `embryo_EmbryoID` format `DYYYY.MM.DD_...`) and a **month drill-down** for every unit × year combination where Local ≠ Athena.

In [4]:
# 1. Location Comparison
df_duck_loc_clean = df_duck_loc.rename(columns={
    'patient_unit_huntington': 'location',
    'total_rows': 'duck_rows',
    'unique_patients': 'duck_patients',
    'unique_patient_idxs': 'duck_patient_idxs',
    'unique_prontuarios': 'duck_prontuarios',
    'pct_patients_with_valid_prontuario': 'duck_valid_pront_pct'
})[['location', 'duck_rows', 'duck_patients', 'duck_patient_idxs', 'duck_prontuarios', 'duck_valid_pront_pct']]

df_ath_loc_clean = df_ath_loc.rename(columns={
    'unit_huntington': 'location',
    'total_rows': 'ath_rows',
    'unique_patients': 'ath_patients',
    'unique_patient_idxs': 'ath_patient_idxs',
    'unique_prontuarios': 'ath_prontuarios',
    'pct_patients_with_valid_prontuario': 'ath_valid_pront_pct'
})[['location', 'ath_rows', 'ath_patients', 'ath_patient_idxs', 'ath_prontuarios', 'ath_valid_pront_pct']]

df_comp_loc = pd.merge(df_duck_loc_clean, df_ath_loc_clean, on='location')
print("--- Side-by-side Location Comparison ---")
display(df_comp_loc)

# 2. Year Comparison
df_duck_yr_clean = df_duck_year.rename(columns={
    'embryo_year': 'year',
    'total_rows': 'duck_rows',
    'unique_patients': 'duck_patients',
    'unique_patient_idxs': 'duck_patient_idxs',
    'unique_prontuarios': 'duck_prontuarios',
    'pct_patients_with_valid_prontuario': 'duck_valid_pront_pct'
})[['year', 'duck_rows', 'duck_patients', 'duck_patient_idxs', 'duck_prontuarios', 'duck_valid_pront_pct']]

df_ath_yr_clean = df_ath_year.rename(columns={
    'embryo_year': 'year',
    'total_rows': 'ath_rows',
    'unique_patients': 'ath_patients',
    'unique_patient_idxs': 'ath_patient_idxs',
    'unique_prontuarios': 'ath_prontuarios',
    'pct_patients_with_valid_prontuario': 'ath_valid_pront_pct'
})[['year', 'ath_rows', 'ath_patients', 'ath_patient_idxs', 'ath_prontuarios', 'ath_valid_pront_pct']]

# Ensure both are integers for seamless merge
df_duck_yr_clean['year'] = df_duck_yr_clean['year'].astype(float).astype(int)
df_ath_yr_clean['year'] = df_ath_yr_clean['year'].astype(float).astype(int)

df_comp_yr = pd.merge(df_duck_yr_clean, df_ath_yr_clean, on='year')
print("\n--- Side-by-side Year Comparison ---")
display(df_comp_yr)


# ── helpers ────────────────────────────────────────────────────────────────
def _extract_year(eid):
    """EmbryoID format: DYYYY.MM.DD_... -> YYYY"""
    if eid is None or (isinstance(eid, float) and pd.isna(eid)):
        return "N/A"
    s = str(eid).strip()
    if len(s) >= 5 and s[0].upper() == 'D' and s[1:5].isdigit():
        return s[1:5]
    return "N/A"

def _extract_yearmonth(eid):
    """EmbryoID format: DYYYY.MM.DD_... -> YYYY-MM"""
    if eid is None or (isinstance(eid, float) and pd.isna(eid)):
        return "N/A"
    s = str(eid).strip()
    if len(s) >= 8 and s[0].upper() == 'D' and s[1:5].isdigit() and s[5] == '.' and s[6:8].isdigit():
        return f"{s[1:5]}-{s[6:8]}"
    return "N/A"

# ── 3. Year × Unit Pivot ────────────────────────────────────────────────────
print("\n" + "=" * 80)
print("📅 Year × Unit Pivot (year from embryo_EmbryoID)")
print("=" * 80)

duck_id_loc = duck_con.execute(
    "SELECT embryo_EmbryoID, patient_unit_huntington FROM gold.embryoscope_embrioes"
).df()
duck_id_loc['year'] = duck_id_loc['embryo_EmbryoID'].apply(_extract_year)
duck_id_loc['year_month'] = duck_id_loc['embryo_EmbryoID'].apply(_extract_yearmonth)

ath_cur.execute(
    "SELECT embryo_embryo_id, unit_huntington FROM gold_huntington_prod.embryoscope_embrioes"
)
ath_id_loc = pd.DataFrame(ath_cur.fetchall(), columns=[d[0] for d in ath_cur.description])
ath_id_loc['year'] = ath_id_loc['embryo_embryo_id'].apply(_extract_year)
ath_id_loc['year_month'] = ath_id_loc['embryo_embryo_id'].apply(_extract_yearmonth)

years = sorted(set(duck_id_loc['year']) | set(ath_id_loc['year']))
units = sorted(set(duck_id_loc['patient_unit_huntington'].dropna()) | set(ath_id_loc['unit_huntington'].dropna()))

def _pivot_rows(df, unit_col, label):
    rows = []
    for yr in years:
        row = {'Year': yr}
        yr_df = df[df['year'] == yr]
        for u in units:
            row[u] = int((yr_df[unit_col] == u).sum())
        row['Total'] = int(len(yr_df))
        rows.append(row)
    return pd.DataFrame(rows).set_index('Year')

duck_pivot = _pivot_rows(duck_id_loc, 'patient_unit_huntington', 'DuckDB')
ath_pivot  = _pivot_rows(ath_id_loc,  'unit_huntington',          'Athena')

print("\n### Local DuckDB — Rows by Year × Unit\n")
display(duck_pivot)
print("\n### Athena Production — Rows by Year × Unit\n")
display(ath_pivot)
print("\n### Delta (Local − Athena)\n")
display(duck_pivot.subtract(ath_pivot, fill_value=0).astype(int))

# ── 4. Month Drill-Down for Discrepant Unit × Year ─────────────────────────
print("\n" + "=" * 80)
print("🔍 Month Drill-Down — only unit × year combos with discrepancies")
print("=" * 80)

found_any = False
for unit in units:
    d_unit = duck_id_loc[duck_id_loc['patient_unit_huntington'] == unit]
    a_unit = ath_id_loc[ath_id_loc['unit_huntington'] == unit]

    for yr in years:
        d_yr_ids = set(d_unit[d_unit['year'] == yr]['embryo_EmbryoID'].dropna())
        a_yr_ids = set(a_unit[a_unit['year'] == yr]['embryo_embryo_id'].dropna())

        if d_yr_ids == a_yr_ids:
            continue  # no difference — skip

        year_months = sorted(
            set(d_unit[d_unit['year'] == yr]['year_month']) |
            set(a_unit[a_unit['year'] == yr]['year_month'])
        )

        monthly_rows = []
        for ym in year_months:
            d_ym_ids = set(d_unit[d_unit['year_month'] == ym]['embryo_EmbryoID'].dropna())
            a_ym_ids = set(a_unit[a_unit['year_month'] == ym]['embryo_embryo_id'].dropna())
            only_d = len(d_ym_ids - a_ym_ids)
            only_a = len(a_ym_ids - d_ym_ids)
            if only_d > 0 or only_a > 0:
                monthly_rows.append({
                    'Year-Month':   ym,
                    'Local Count':  len(d_ym_ids),
                    'Athena Count': len(a_ym_ids),
                    'Matched':      len(d_ym_ids & a_ym_ids),
                    'Only Local':   only_d,
                    'Only Athena':  only_a,
                })

        if monthly_rows:
            found_any = True
            print("=" * 70)
            print(f"  Unit: {unit}  |  Year: {yr}")
            print("=" * 70)
            display(pd.DataFrame(monthly_rows).set_index('Year-Month'))
            print()

if not found_any:
    print("No discrepancies found across any unit × year combination.")


--- Side-by-side Location Comparison ---


,location,duck_rows,duck_patients,duck_patient_idxs,duck_prontuarios,duck_valid_pront_pct,ath_rows,ath_patients,ath_patient_idxs,ath_prontuarios,ath_valid_pront_pct
0,Belo Horizonte,27304,2562,2562,2448,99.02,27304,2562,2562,2448,99.02
1,Brasilia,18478,1791,1793,1747,98.49,18478,1791,1793,1746,98.44
2,Ibirapuera,66859,6027,6154,5952,99.75,66809,6023,6150,5948,99.75
3,Salvador,2315,246,246,241,97.56,2315,246,246,242,97.97
4,Vila Mariana,37961,3339,3350,3301,99.64,37961,3339,3350,3301,99.64



--- Side-by-side Year Comparison ---


,year,duck_rows,duck_patients,duck_patient_idxs,duck_prontuarios,duck_valid_pront_pct,ath_rows,ath_patients,ath_patient_idxs,ath_prontuarios,ath_valid_pront_pct
0,2017,602,81,81,81,100.00,602,81,81,81,100.00
1,2018,4127,455,458,455,99.78,4127,455,458,455,99.78
2,2019,14459,1642,1671,1638,99.39,14459,1642,1671,1638,99.39
3,2020,17348,1798,1823,1775,99.17,17348,1798,1823,1774,99.11
4,2021,21132,2297,2345,2258,99.00,21132,2297,2345,2258,99.00
5,2022,19712,2101,2144,2043,99.43,19712,2101,2144,2043,99.43
6,2023,19215,1985,2008,1963,99.45,19215,1985,2008,1962,99.40
7,2024,17282,1760,1776,1752,99.77,17282,1760,1776,1752,99.77
8,2025,23570,2358,2397,2340,99.58,23527,2354,2393,2338,99.66
9,2026,15470,1581,1596,1574,99.81,15463,1581,1596,1574,99.81



📅 Year × Unit Pivot (year from embryo_EmbryoID)

### Local DuckDB — Rows by Year × Unit



...

,Belo Horizonte,Brasilia,Ibirapuera,Salvador,Vila Mariana,Total
Year,,,,,,
2017,0,0,0,0,0,0
2018,0,0,0,0,0,0
2019,0,0,0,0,0,0
2020,0,0,0,0,0,0
2021,0,0,0,0,0,0
2022,0,0,0,0,0,0
2023,0,0,0,0,0,0
2024,0,0,0,0,0,0
2025,0,0,43,0,0,43



🔍 Month Drill-Down — only unit × year combos with discrepancies
  Unit: Ibirapuera  |  Year: 2025


,Local Count,Athena Count,Matched,Only Local,Only Athena
Year-Month,,,,,
2025-02,631,616,616,15,0
2025-03,610,601,601,9,0
2025-05,749,742,742,7,0
2025-09,802,794,794,8,0
2025-10,744,740,740,4,0



  Unit: Ibirapuera  |  Year: 2026


,Local Count,Athena Count,Matched,Only Local,Only Athena
Year-Month,,,,,
2026-07,773,766,766,7,0


## 🔍 Part 4: Discrepant Patient Sample Comparison (First Record)
This section finds 5 patients who have a valid `prontuario` in DuckDB, but have a missing/default (`NULL` or `-1`) `prontuario` in AWS Athena. It retrieves and displays the first record (line) for each of these 5 patients from both sources for direct comparison.

In [5]:
# 1. Fetch unique patients (at patient_PatientIDx level) and their prontuarios
df_duck_pts = duck_con.execute("""
    SELECT DISTINCT patient_PatientIDx, patient_PatientID, patient_FirstName as first_name, prontuario
    FROM gold.embryoscope_embrioes
    WHERE prontuario IS NOT NULL AND prontuario <> -1 AND CAST(prontuario AS VARCHAR) <> '-1'
""").df()

ath_cur.execute("""
    SELECT DISTINCT patient_id_x, patient_id, first_name, prontuario
    FROM gold_huntington_prod.embryoscope_embrioes
""")
df_ath_pts = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])

# Convert IDxs and IDs to string to prevent merge type mismatch
df_duck_pts['patient_PatientIDx'] = df_duck_pts['patient_PatientIDx'].astype(str)
df_ath_pts['patient_id_x'] = df_ath_pts['patient_id_x'].astype(str)

# 2. Find discrepancies where Athena prontuario is NULL or -1, joining on patient_id_x
merged_pts = pd.merge(df_duck_pts, df_ath_pts, left_on='patient_PatientIDx', right_on='patient_id_x', suffixes=('_duck', '_ath'))
discrepant = merged_pts[
    merged_pts['prontuario_ath'].isna() | 
    (merged_pts['prontuario_ath'] == -1) | 
    (merged_pts['prontuario_ath'].astype(str) == '-1') | 
    (merged_pts['prontuario_ath'].astype(str) == 'None')
].drop_duplicates(subset=['patient_PatientIDx']).head(5)

sample_patient_idxs = discrepant['patient_PatientIDx'].tolist()
print("Discrepant Patient IDxs identified:", sample_patient_idxs)

# 3. Fetch all records for only these 5 patients
df_duck_all_records = duck_con.execute(f"""
    SELECT prontuario, patient_PatientID, patient_PatientIDx, patient_FirstName as first_name, patient_unit_huntington, embryo_EmbryoID, embryo_EmbryoDate
    FROM gold.embryoscope_embrioes
    WHERE CAST(patient_PatientIDx AS VARCHAR) IN ({','.join("'" + px + "'" for px in sample_patient_idxs)})
""").df()
df_duck_all_records['patient_PatientIDx'] = df_duck_all_records['patient_PatientIDx'].astype(str)

ath_cur.execute(f"""
    SELECT prontuario, patient_id, patient_id_x, first_name, unit_huntington, embryo_embryo_id, embryo_embryo_date
    FROM gold_huntington_prod.embryoscope_embrioes
    WHERE patient_id_x IN ({','.join("'" + px + "'" for px in sample_patient_idxs)})
""")
df_ath_all_records = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
df_ath_all_records['patient_id_x'] = df_ath_all_records['patient_id_x'].astype(str)

# 4. Display the first record for each patient in each database (sorted by patient_id_x)
df_duck_first = df_duck_all_records.drop_duplicates(subset=['patient_PatientIDx']).sort_values(by='patient_PatientIDx')
df_ath_first = df_ath_all_records.drop_duplicates(subset=['patient_id_x']).sort_values(by='patient_id_x')

print("\n🔹 [DuckDB] First line for each of the 5 discrepant patients (Sorted):")
display(df_duck_first)

print("\n🔸 [Athena Prod] First line for each of the 5 discrepant patients (Sorted):")
display(df_ath_first)

# 5. Query spouse names from Athena silver_clinisys_prod.view_pacientes using DuckDB's valid prontuarios
print("\n🌸 [Athena silver_clinisys_prod] Wife & Husband names matching DuckDB prontuarios:")
sample_prontuarios = df_duck_first['prontuario'].dropna().tolist()
sample_prontuarios_clean = [int(p) for p in sample_prontuarios if str(p) != '-1' and pd.notnull(p)]

if sample_prontuarios_clean:
    pront_list_str = ','.join(str(p) for p in sample_prontuarios_clean)
    clinisys_q = f"""
        SELECT codigo as prontuario, esposa_nome, marido_nome 
        FROM silver_clinisys_prod.view_pacientes 
        WHERE codigo IN ({pront_list_str})
    """
    ath_cur.execute(clinisys_q)
    df_clinisys_names = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
    df_clinisys_names['prontuario'] = df_clinisys_names['prontuario'].astype(str)
    display(df_clinisys_names)
else:
    print("No valid prontuario values available to query Clinisys spouse names.")

Discrepant Patient IDxs identified: ['S4JP1389_45117.4839894560', 'NEXTGEN_46040.2969325463', 'S4JP138913613_43964.5109041435']

🔹 [DuckDB] First line for each of the 5 discrepant patients (Sorted):


,prontuario,patient_PatientID,patient_PatientIDx,first_name,patient_unit_huntington,embryo_EmbryoID,embryo_EmbryoDate
168,185412,185412,NEXTGEN_46040.2969325463,"CASELLA, M. FERNANDA D.P.",Ibirapuera,D2026.01.18_S04721_I3166_P-5,2026-01-18
8,762312,2312,S4JP138913613_43964.5109041435,"Santos,",Brasilia,D2020.05.19_S00053_I4120_P-13,2020-05-19
0,811383,811383,S4JP1389_45117.4839894560,"QUEIROZ, R. KELLY",Brasilia,D2023.07.10_S01873_I4120_P-1,2023-07-10



🔸 [Athena Prod] First line for each of the 5 discrepant patients (Sorted):


,prontuario,patient_id,patient_id_x,first_name,unit_huntington,embryo_embryo_id,embryo_embryo_date
168,-1,185412,NEXTGEN_46040.2969325463,"CASELLA, M. FERNANDA D.P.",Ibirapuera,D2026.01.18_S04721_I3166_P-5,2026-01-18
0,-1,2312,S4JP138913613_43964.5109041435,"Santos,",Brasilia,D2020.05.28_S00060_I4120_P-16,2020-05-28
160,-1,811383,S4JP1389_45117.4839894560,"QUEIROZ, R. KELLY",Brasilia,D2023.07.10_S01873_I4120_P-8,2023-07-10



🌸 [Athena silver_clinisys_prod] Wife & Husband names matching DuckDB prontuarios:


,prontuario,esposa_nome,marido_nome
0,762312,MARINA DOS SANTOS FERREIRA,None
1,811383,Kelly Rocha de Queiroz,Guilheme Paiva Silva
2,185412,Maria Fernanda Durao Pelis Casella,Caio Baeta Casella


## 📑 Part 5: Discrepancy & Exclusive Row Analysis
This section calculates the total count of discrepant embryo records (`embryo_EmbryoID` / `embryo_embryo_id`) that exist exclusively in one database but not the other, applying the sync lag and server outage filters. It displays the total discrepancy volumes and a sample of the exclusive records.

In [6]:
# 1. Fetch all records from both databases with the reporting column set
print("Fetching all records from DuckDB...")
df_duck_all = duck_con.execute("""
    SELECT prontuario, patient_PatientID, patient_PatientIDx, patient_FirstName as first_name, patient_unit_huntington, embryo_EmbryoID, embryo_EmbryoDate
    FROM gold.embryoscope_embrioes
""").df()
df_duck_all['embryo_EmbryoID'] = df_duck_all['embryo_EmbryoID'].astype(str)

print("Fetching all records from AWS Athena Prod...")
ath_cur.execute("""
    SELECT prontuario, patient_id, patient_id_x, first_name, unit_huntington, embryo_embryo_id, embryo_embryo_date
    FROM gold_huntington_prod.embryoscope_embrioes
""")
df_ath_all = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
df_ath_all['embryo_embryo_id'] = df_ath_all['embryo_embryo_id'].astype(str)

# Get max dates for sync lag verification
max_ath_date = pd.to_datetime(df_ath_all['embryo_embryo_date']).max()
max_duck_date = pd.to_datetime(df_duck_all['embryo_EmbryoDate']).max()

# Convert to datetime for filtering
df_duck_all['datetime_EmbryoDate'] = pd.to_datetime(df_duck_all['embryo_EmbryoDate'])
df_ath_all['datetime_embryo_date'] = pd.to_datetime(df_ath_all['embryo_embryo_date'])

# Print active filters
# print("\n⚠️ Discrepancy comparison filters applied:")
# print("  - Ignored Ibirapuera records from 2025 (known server issue)")
print("  - Filtered out records at or after the max embryo date of the target database (sync lag buffer)")
print(f"  - DuckDB max embryo date: {max_duck_date.strftime('%Y-%m-%d') if pd.notnull(max_duck_date) else 'N/A'}")
print(f"  - AWS Athena max embryo date: {max_ath_date.strftime('%Y-%m-%d') if pd.notnull(max_ath_date) else 'N/A'}")

# 2. Find records ONLY in DuckDB (missing in Athena)
df_only_duck = df_duck_all[~df_duck_all['embryo_EmbryoID'].isin(df_ath_all['embryo_embryo_id'])].copy()
# df_only_duck = df_only_duck[~(
#     (df_only_duck['patient_unit_huntington'] == 'Ibirapuera') &
#     (df_only_duck['datetime_EmbryoDate'].dt.year == 2025)
# )]
df_only_duck = df_only_duck[df_only_duck['datetime_EmbryoDate'] < max_ath_date]

print(f"\n🔴 Total records only in DuckDB: {len(df_only_duck)}")
if not df_only_duck.empty:
    print("Sample of records only in DuckDB (Filtered):")
    display(df_only_duck.head(10))
else:
    print("No exclusive DuckDB records found matching the restrictions.")

# 3. Find records ONLY in AWS Athena Prod (missing in DuckDB)
df_only_ath = df_ath_all[~df_ath_all['embryo_embryo_id'].isin(df_duck_all['embryo_EmbryoID'])].copy()
# df_only_ath = df_only_ath[~(
#     (df_only_ath['unit_huntington'] == 'Ibirapuera') &
#     (df_only_ath['datetime_embryo_date'].dt.year == 2025)
# )]
df_only_ath = df_only_ath[df_only_ath['datetime_embryo_date'] < max_duck_date]

print(f"\n🔵 Total records only in AWS Athena Prod: {len(df_only_ath)}")
if not df_only_ath.empty:
    print("Sample of records only in AWS Athena Prod (Filtered):")
    display(df_only_ath.head(10))
else:
    print("No exclusive Athena records found matching the restrictions.")

Fetching all records from DuckDB...
Fetching all records from AWS Athena Prod...
  - Filtered out records at or after the max embryo date of the target database (sync lag buffer)
  - DuckDB max embryo date: 2026-08-16
  - AWS Athena max embryo date: 2026-08-16

🔴 Total records only in DuckDB: 50
Sample of records only in DuckDB (Filtered):


,prontuario,patient_PatientID,patient_PatientIDx,first_name,patient_unit_huntington,embryo_EmbryoID,embryo_EmbryoDate,datetime_EmbryoDate
89989,920117,920117,PC1P7BHG_46198.4969735185,"TAKATA, ALINE",Ibirapuera,D2026.07.25_S05076_I3166_P-7,2026-07-25,2026-07-25
90610,756121,756121,PC1P7BHG_46168.4420869792,"APPEZZATO, HELENA E. C.",Ibirapuera,D2026.07.27_S04799_I3027_P-10,2026-07-27,2026-07-27
90900,171395,171395,PC1P7BHG_46155.4248835764,"SALVINI, IRACEMA S.",Ibirapuera,D2026.07.29_S05090_I3166_P-7,2026-07-29,2026-07-29
92363,898239,898239,PC1P7BHG_46108.4636951968,"MORAES, AMANDA C. M.",Ibirapuera,D2026.07.29_S05089_I3166_P-5,2026-07-29,2026-07-29
92489,908334,908334,PC1P7BHG_46104.5466231597,"OLIVEIRA, LIVIA S.",Ibirapuera,D2026.07.27_S04803_I3027_P-7,2026-07-27,2026-07-27
93305,900381,900381,PC1P7BHG_46076.5156899653,"DANTAS, ELANE F.",Ibirapuera,D2026.07.27_S04800_I3027_P-10,2026-07-27,2026-07-27
93797,177072,177072,PC1P7BHG_46056.4528425926,"DISI, ILANA R.",Ibirapuera,D2026.07.27_S04798_I3027_P-7,2026-07-27,2026-07-27
100880,865375,865375,PC1P7BHG_45702.3600089815,"OLIVEIRA, MARIANA C. L. M.",Ibirapuera,D2025.02.14_S04038_I3166_P-2,2025-02-14,2025-02-14
100881,865375,865375,PC1P7BHG_45702.3600089815,"OLIVEIRA, MARIANA C. L. M.",Ibirapuera,D2025.02.14_S04038_I3166_P-1,2025-02-14,2025-02-14
100882,865375,865375,PC1P7BHG_45702.3600089815,"OLIVEIRA, MARIANA C. L. M.",Ibirapuera,D2025.02.14_S04038_I3166_P-3,2025-02-14,2025-02-14



🔵 Total records only in AWS Athena Prod: 0
No exclusive Athena records found matching the restrictions.


In [7]:
df_only_duck[df_only_duck['patient_unit_huntington'] == 'Vila Mariana']

,prontuario,patient_PatientID,patient_PatientIDx,first_name,patient_unit_huntington,embryo_EmbryoID,embryo_EmbryoDate,datetime_EmbryoDate


In [8]:
df_only_ath[df_only_ath['unit_huntington']=="Ibirapuera"].shape

(0, 8)

In [9]:
df_only_ath[df_only_ath['unit_huntington']=='Ibirapuera']

,prontuario,patient_id,patient_id_x,first_name,unit_huntington,embryo_embryo_id,embryo_embryo_date,datetime_embryo_date


## 🥚 Part 6: Embryo Number Mismatch Analysis
This section identifies embryo records that exist in **both** databases (matched by embryo ID) but have **different** values for the embryo number field (`embryo_embryo_number` in DuckDB vs. `embryo_number` in AWS Athena).

In [10]:
# 1. Fetch reporting column set plus embryo numbers from both databases
print("Fetching embryo records with embryo numbers from DuckDB...")
df_duck_num = duck_con.execute("""
    SELECT prontuario, patient_PatientID, patient_PatientIDx, patient_FirstName as first_name, patient_unit_huntington, embryo_EmbryoID, embryo_EmbryoDate, embryo_embryo_number
    FROM gold.embryoscope_embrioes
""").df()
df_duck_num['embryo_EmbryoID'] = df_duck_num['embryo_EmbryoID'].astype(str)

print("Fetching embryo records with embryo numbers from AWS Athena Prod...")
ath_cur.execute("""
    SELECT prontuario, patient_id, patient_id_x, first_name, unit_huntington, embryo_embryo_id, embryo_embryo_date, embryo_number
    FROM gold_huntington_prod.embryoscope_embrioes
""")
df_ath_num = pd.DataFrame(ath_cur.fetchall(), columns=[desc[0] for desc in ath_cur.description])
df_ath_num['embryo_embryo_id'] = df_ath_num['embryo_embryo_id'].astype(str)

# 2. Join the databases on embryo ID (inner join to find records existing in both)
df_merged_num = pd.merge(
    df_duck_num, 
    df_ath_num, 
    left_on='embryo_EmbryoID', 
    right_on='embryo_embryo_id', 
    suffixes=('_duck', '_ath')
)

# 3. Clean and convert embryo numbers to numeric to ensure clean comparison
df_merged_num['embryo_num_clean_duck'] = pd.to_numeric(df_merged_num['embryo_embryo_number'], errors='coerce')
df_merged_num['embryo_num_clean_ath'] = pd.to_numeric(df_merged_num['embryo_number'], errors='coerce')

# 4. Find records where clean embryo numbers do not match
df_mismatched_num = df_merged_num[
    df_merged_num['embryo_num_clean_duck'] != df_merged_num['embryo_num_clean_ath']
]

print(f"\n⚠️ Total matched embryo records with differing embryo numbers: {len(df_mismatched_num)}")
if not df_mismatched_num.empty:
    print("Sample of mismatched embryo number records:")
    # Display clean selection showing comparative columns
    display_cols = [
        'embryo_EmbryoID', 'patient_PatientIDx', 'first_name_duck', 
        'embryo_embryo_number', 'embryo_number', 
        'patient_unit_huntington', 'unit_huntington'
    ]
    display(df_mismatched_num[display_cols].head(10))
else:
    print("No embryo number discrepancies found for overlapping embryo records.")

Fetching embryo records with embryo numbers from DuckDB...
Fetching embryo records with embryo numbers from AWS Athena Prod...

⚠️ Total matched embryo records with differing embryo numbers: 6
Sample of mismatched embryo number records:


,embryo_EmbryoID,patient_PatientIDx,first_name_duck,embryo_embryo_number,embryo_number,patient_unit_huntington,unit_huntington
43119,D2025.02.17_S03960_I3027_P-1,S4AY087741396_43717.5465543171,"BORGES, DANIELLE W. M.",18,1,Ibirapuera,Ibirapuera
102332,D2025.02.15_S04042_I3166_P-1,PC1P7BHG_45604.5450327083,"GIESTA, LARISSA L. DA S.",3,1,Ibirapuera,Ibirapuera
102333,D2025.02.15_S04042_I3166_P-2,PC1P7BHG_45604.5450327083,"GIESTA, LARISSA L. DA S.",4,2,Ibirapuera,Ibirapuera
103752,D2025.02.19_S03968_I3027_P-3,PC1P7BHG_45520.4332258796,"COSTA, TATIANA Z.",3,1,Ibirapuera,Ibirapuera
103753,D2025.02.19_S03968_I3027_P-10,PC1P7BHG_45520.4332258796,"COSTA, TATIANA Z.",4,2,Ibirapuera,Ibirapuera
103754,D2025.02.19_S03968_I3027_P-11,PC1P7BHG_45520.4332258796,"COSTA, TATIANA Z.",5,3,Ibirapuera,Ibirapuera


## 🔬 Part 6: Morphological & Timing Annotations Completeness
Compares the non-null filling rates (%) of key embryology morphokinetics (`t2`, `t5`, `t8`, `tB`, `ICM`, `TE`) between local DuckDB and AWS Athena Prod.

In [11]:
duck_ann = duck_con.execute("""
    SELECT 
        COUNT(*) as total_rows,
        ROUND(COUNT(embryo_Time_t2)*100.0/COUNT(*), 2) as t2_pct,
        ROUND(COUNT(embryo_Time_t5)*100.0/COUNT(*), 2) as t5_pct,
        ROUND(COUNT(embryo_Time_t8)*100.0/COUNT(*), 2) as t8_pct,
        ROUND(COUNT(embryo_Time_tB)*100.0/COUNT(*), 2) as tb_pct,
        ROUND(COUNT(embryo_Value_ICM)*100.0/COUNT(*), 2) as icm_pct,
        ROUND(COUNT(embryo_Value_TE)*100.0/COUNT(*), 2) as te_pct
    FROM gold.embryoscope_embrioes
""").df()

ath_cur.execute("""
    SELECT 
        COUNT(*) as total_rows,
        ROUND(COUNT(embryo_time_t2)*100.0/COUNT(*), 2) as t2_pct,
        ROUND(COUNT(embryo_time_t5)*100.0/COUNT(*), 2) as t5_pct,
        ROUND(COUNT(embryo_time_t8)*100.0/COUNT(*), 2) as t8_pct,
        ROUND(COUNT(embryo_time_tb)*100.0/COUNT(*), 2) as tb_pct,
        ROUND(COUNT(embryo_value_icm)*100.0/COUNT(*), 2) as icm_pct,
        ROUND(COUNT(embryo_value_te)*100.0/COUNT(*), 2) as te_pct
    FROM gold_huntington_prod.embryoscope_embrioes
""")
ath_ann = pd.DataFrame(ath_cur.fetchall(), columns=[d[0] for d in ath_cur.description])

df_ann_comp = pd.DataFrame({
    'annotation_field': ['t2_pct', 't5_pct', 't8_pct', 'tb_pct', 'icm_pct', 'te_pct'],
    'duck_fill_pct': [duck_ann[c].iloc[0] for c in ['t2_pct', 't5_pct', 't8_pct', 'tb_pct', 'icm_pct', 'te_pct']],
    'ath_fill_pct': [ath_ann[c].iloc[0] for c in ['t2_pct', 't5_pct', 't8_pct', 'tb_pct', 'icm_pct', 'te_pct']]
})
df_ann_comp['diff_pct'] = (df_ann_comp['duck_fill_pct'] - df_ann_comp['ath_fill_pct']).round(2)
print("--- Annotations Completion Rate Comparison (%) ---")
display(df_ann_comp)

--- Annotations Completion Rate Comparison (%) ---


,annotation_field,duck_fill_pct,ath_fill_pct,diff_pct
0,t2_pct,51.91,51.89,0.02
1,t5_pct,47.92,47.90,0.02
2,t8_pct,44.44,44.42,0.02
3,tb_pct,36.64,36.62,0.02
4,icm_pct,34.97,34.94,0.03
5,te_pct,34.94,34.91,0.03


## ❄️ Part 7: Embryo Fate Categorical Breakdown
Compares row distributions per `embryo_EmbryoFate` categorical value between local DuckDB and AWS Athena Prod.

In [12]:
duck_fate = duck_con.execute("SELECT embryo_EmbryoFate as fate, COUNT(*) as duck_count FROM gold.embryoscope_embrioes GROUP BY 1").df()
ath_cur.execute("SELECT embryo_embryo_fate as fate, COUNT(*) as ath_count FROM gold_huntington_prod.embryoscope_embrioes GROUP BY 1")
ath_fate = pd.DataFrame(ath_cur.fetchall(), columns=[d[0] for d in ath_cur.description])

merged_fate = pd.merge(duck_fate, ath_fate, on='fate', how='outer').fillna(0)
merged_fate['duck_count'] = merged_fate['duck_count'].astype(int)
merged_fate['ath_count'] = merged_fate['ath_count'].astype(int)
merged_fate['diff_count'] = merged_fate['duck_count'] - merged_fate['ath_count']
print("--- Embryo Fate Distribution Comparison ---")
display(merged_fate.sort_values(by='duck_count', ascending=False))

--- Embryo Fate Distribution Comparison ---


,fate,duck_count,ath_count,diff_count
0,Avoid,83852,83843,9
1,Freeze,48085,48122,-37
5,Unknown,9854,9824,30
2,FrozenEmbryoTransfer,7155,7109,46
3,Transfer,3324,3323,1
4,Undecided,647,646,1


## 📅 Part 8: Monthly Volume Trend (`YYYY-MM`)
Displays monthly row counts for the recent 12 months to highlight exact sync lag thresholds.

In [13]:
duck_month = duck_con.execute("SELECT strftime(embryo_EmbryoDate, '%Y-%m') as year_month, COUNT(*) as duck_rows FROM gold.embryoscope_embrioes GROUP BY 1 ORDER BY 1").df()
ath_cur.execute("SELECT date_format(embryo_embryo_date, '%Y-%m') as year_month, COUNT(*) as ath_rows FROM gold_huntington_prod.embryoscope_embrioes GROUP BY 1 ORDER BY 1")
ath_month = pd.DataFrame(ath_cur.fetchall(), columns=[d[0] for d in ath_cur.description])

merged_month = pd.merge(duck_month, ath_month, on='year_month', how='outer').fillna(0)
merged_month['duck_rows'] = merged_month['duck_rows'].astype(int)
merged_month['ath_rows'] = merged_month['ath_rows'].astype(int)
merged_month['diff_rows'] = merged_month['duck_rows'] - merged_month['ath_rows']

print("--- Recent 12-Month Volume Trend ---")
display(merged_month.sort_values(by='year_month', ascending=False).head(12))

--- Recent 12-Month Volume Trend ---


,year_month,duck_rows,ath_rows,diff_rows
106,2026-08,1024,1024,0
105,2026-07,2093,2086,7
104,2026-06,2025,2025,0
103,2026-05,2474,2474,0
102,2026-04,1962,1962,0
101,2026-03,2018,2018,0
100,2026-02,1688,1688,0
99,2026-01,2186,2186,0
98,2025-12,1291,1291,0
97,2025-11,2309,2309,0


## 🧮 Part 9: Numeric Score & Age Reconciliation (`AgeAtFertilization`)
Reconciles `AgeAtFertilization` numeric values across matched embryo IDs in both databases to verify float precision.

In [14]:
duck_age = duck_con.execute("SELECT embryo_EmbryoID, AgeAtFertilization FROM gold.embryoscope_embrioes WHERE AgeAtFertilization IS NOT NULL").df()
ath_cur.execute("SELECT embryo_embryo_id, age_at_fertilization FROM gold_huntington_prod.embryoscope_embrioes WHERE age_at_fertilization IS NOT NULL")
ath_age = pd.DataFrame(ath_cur.fetchall(), columns=[d[0] for d in ath_cur.description])

duck_age['embryo_EmbryoID'] = duck_age['embryo_EmbryoID'].astype(str)
ath_age['embryo_embryo_id'] = ath_age['embryo_embryo_id'].astype(str)

merged_age = pd.merge(duck_age, ath_age, left_on='embryo_EmbryoID', right_on='embryo_embryo_id')
merged_age['abs_diff'] = (merged_age['AgeAtFertilization'] - merged_age['age_at_fertilization']).abs()
exact_matches = (merged_age['abs_diff'] < 0.01).sum()
total_matched = len(merged_age)

print(f"Total Matched Embryos Audited: {total_matched}")
print(f"Exact Age Matches (<0.01 diff): {exact_matches} ({exact_matches*100.0/total_matched:.2f}%)")
if exact_matches < total_matched:
    print("\nSample of numeric age discrepancies:")
    display(merged_age[merged_age['abs_diff'] >= 0.01].head(5))

Total Matched Embryos Audited: 147310
Exact Age Matches (<0.01 diff): 147194 (99.92%)

Sample of numeric age discrepancies:


,embryo_EmbryoID,AgeAtFertilization,embryo_embryo_id,age_at_fertilization,abs_diff
22963,D2018.10.21_S00592_I3027_P-10,33.66,D2018.10.21_S00592_I3027_P-10,33.72,0.06
22964,D2018.10.21_S00592_I3027_P-11,33.66,D2018.10.21_S00592_I3027_P-11,33.72,0.06
22965,D2018.10.21_S00592_I3027_P-12,33.66,D2018.10.21_S00592_I3027_P-12,33.72,0.06
22966,D2019.02.23_S00117_I3166_P-12,34.00,D2019.02.23_S00117_I3166_P-12,34.06,0.06
22967,D2019.02.23_S00117_I3166_P-14,34.00,D2019.02.23_S00117_I3166_P-14,34.06,0.06


## 🔗 Part 10: Patient ID Mapping Integrity (`patient_id` vs `patient_id_x`)
Audits the number of unique `patient_id_x` treatment records per `patient_id`.

In [15]:
duck_integrity = duck_con.execute("SELECT patient_PatientID as patient_id, COUNT(DISTINCT patient_PatientIDx) as idx_count_duck FROM gold.embryoscope_embrioes GROUP BY 1").df()
ath_cur.execute("SELECT patient_id, COUNT(DISTINCT patient_id_x) as idx_count_ath FROM gold_huntington_prod.embryoscope_embrioes GROUP BY 1")
ath_integrity = pd.DataFrame(ath_cur.fetchall(), columns=[d[0] for d in ath_cur.description])

duck_integrity['patient_id'] = duck_integrity['patient_id'].astype(str)
ath_integrity['patient_id'] = ath_integrity['patient_id'].astype(str)

merged_integrity = pd.merge(duck_integrity, ath_integrity, on='patient_id', how='outer').fillna(0)
merged_integrity['idx_count_duck'] = merged_integrity['idx_count_duck'].astype(int)
merged_integrity['idx_count_ath'] = merged_integrity['idx_count_ath'].astype(int)

print("Top patients with multiple patient_id_x treatment references in DuckDB:")
display(merged_integrity.sort_values(by='idx_count_duck', ascending=False).head(10))

Top patients with multiple patient_id_x treatment references in DuckDB:


,patient_id,idx_count_duck,idx_count_ath
1539,2353,4,4
1066,185412,3,3
333,156766,3,3
511,166948,3,3
1491,224143,3,3
1371,222090,3,3
1272,220267,3,3
3632,520607,3,3
1184,213010,3,3
9255,814132,3,3


In [16]:
# Close database connections
print("Closing database connections...")
try:
    duck_con.close()
    print("DuckDB connection closed.")
except Exception as e:
    print(f"Error closing DuckDB connection: {e}")

try:
    ath_con.close()
    print("Athena connection closed.")
except Exception as e:
    print(f"Error closing Athena connection: {e}")

Closing database connections...
DuckDB connection closed.
Athena connection closed.
